## 🔑 API Key Setup

This notebook requires an OpenAI API key.

When you run it, you will be prompted to enter your key securely (input is hidden and not stored).

In [6]:
!pip install openai faiss-cpu numpy pandas pypdf tqdm

In [ ]:
from getpass import getpass
from openai import OpenAI

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Complete lecture notes.pdf to Complete lecture notes.pdf


In [ ]:
documents = []

for filename in uploaded.keys():
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)

        full_text = ""
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                full_text += page_text + "\n"

        documents.append({
            "source": filename,
            "text": full_text
        })

print(f"Loaded {len(documents)} PDF documents.")
for doc in documents:
    print(doc["source"], "->", len(doc["text"]), "characters")

Loaded 1 PDF documents.
Complete lecture notes.pdf -> 707279 characters


In [ ]:
for doc in documents:
    print("=" * 80)
    print("SOURCE:", doc["source"])
    print(doc["text"][:1500])
    print("\n")

SOURCE: Complete lecture notes.pdf
Lecture Notes, FS25
Classical and Bayesian Statistics
Peter Büchel
Lucerne University of Applied Sciences and Arts, Engineering and Architecture

Contents
Preface 1
I. Basics 4
1. Introduction 5
1.1. What is “Statistics”? . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5
1.2. What is “ Applied”Statistics? . . . . . . . . . . . . . . . . . . . . . . . . 6
1.2.1. Problem solving in applied statistics . . . . . . . . . . . . . . . 9
1.2.2. What is Statistics that is not Applied? . . . . . . . . . . . . . . 12
1.3. Classical vs. Bayesian Statistics . . . . . . . . . . . . . . . . . . . . . . 12
1.3.1. Classical Statistics . . . . . . . . . . . . . . . . . . . . . . . . . . 13
1.3.2. Bayesian Statistic . . . . . . . . . . . . . . . . . . . . . . . . . . 14
1.4. Models . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 17
1.5. Simulation, Sampling . . . . . . . . . . . . . . . . . . . . . . . . . . . . 22
2. Exploratory Data Ana

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Text cleaned.")

Text cleaned.


In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

In [ ]:
chunked_docs = []

for doc in documents:
    chunks = chunk_text(doc["text"], chunk_size=800, overlap=150)

    for i, chunk in enumerate(chunks):
        chunked_docs.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": chunk
        })

print(f"Created {len(chunked_docs)} chunks.")
print(chunked_docs[0])

Created 1088 chunks.
{'source': 'Complete lecture notes.pdf', 'chunk_id': 0, 'text': 'Lecture Notes, FS25 Classical and Bayesian Statistics Peter Büchel Lucerne University of Applied Sciences and Arts, Engineering and Architecture Contents Preface 1 I. Basics 4 1. Introduction 5 1.1. What is “Statistics”? . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5 1.2. What is “ Applied”Statistics? . . . . . . . . . . . . . . . . . . . . . . . . 6 1.2.1. Problem solving in applied statistics . . . . . . . . . . . . . . . 9 1.2.2. What is Statistics that is not Applied? . . . . . . . . . . . . . . 12 1.3. Classical vs. Bayesian Statistics . . . . . . . . . . . . . . . . . . . . . . 12 1.3.1. Classical Statistics . . . . . . . . . . . . . . . . . . . . . . . . . . 13 1.3.2. Bayesian Statistic . . . . . . . . . . . . . . . . . . . . . . . . . . 14 1.4. Models . . . . . . . . . .'}


In [ ]:
for i in range(min(3, len(chunked_docs))):
    print("=" * 80)
    print("SOURCE:", chunked_docs[i]["source"])
    print("CHUNK ID:", chunked_docs[i]["chunk_id"])
    print("TEXT:", chunked_docs[i]["text"])
    print("\n")

SOURCE: Complete lecture notes.pdf
CHUNK ID: 0
TEXT: Lecture Notes, FS25 Classical and Bayesian Statistics Peter Büchel Lucerne University of Applied Sciences and Arts, Engineering and Architecture Contents Preface 1 I. Basics 4 1. Introduction 5 1.1. What is “Statistics”? . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5 1.2. What is “ Applied”Statistics? . . . . . . . . . . . . . . . . . . . . . . . . 6 1.2.1. Problem solving in applied statistics . . . . . . . . . . . . . . . 9 1.2.2. What is Statistics that is not Applied? . . . . . . . . . . . . . . 12 1.3. Classical vs. Bayesian Statistics . . . . . . . . . . . . . . . . . . . . . . 12 1.3.1. Classical Statistics . . . . . . . . . . . . . . . . . . . . . . . . . . 13 1.3.2. Bayesian Statistic . . . . . . . . . . . . . . . . . . . . . . . . . . 14 1.4. Models . . . . . . . . . .


SOURCE: Complete lecture notes.pdf
CHUNK ID: 1
TEXT:  . . . . . . . . . . . . . . . . . 13 1.3.2. Bayesian Statistic . . . . . . . . . . . . . .

In [ ]:
texts = [doc["text"] for doc in chunked_docs]

embeddings = []

for text in tqdm(texts):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[text]
    )
    embeddings.append(response.data[0].embedding)

print(f"Created {len(embeddings)} embeddings.")

100%|██████████| 1088/1088 [04:32<00:00,  3.99it/s]

Created 1088 embeddings.


In [ ]:
embedding_matrix = np.array(embeddings).astype("float32")

dimension = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embedding_matrix)

print("FAISS index built successfully.")
print("Number of vectors in index:", index.ntotal)

FAISS index built successfully.
Number of vectors in index: 1088


In [ ]:
def retrieve_chunks(query, k=3):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[query]
    )

    query_embedding = np.array([response.data[0].embedding]).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []
    for idx in indices[0]:
        results.append(chunked_docs[idx])

    return results

In [ ]:
test_results = retrieve_chunks("What is “Statistics”?", k=3)

for r in test_results:
    print("=" * 80)
    print("SOURCE:", r["source"])
    print("CHUNK ID:", r["chunk_id"])
    print("TEXT:", r["text"])
    print("\n")

SOURCE: Complete lecture notes.pdf
CHUNK ID: 60
TEXT: tics is the art of data handling . Collecting data used to be a hassle but that is history. With the advent of computer and cheap sophisticated devices to collect data, such as taking pictures with your smart phone, data are everywhere. Therefore, statistics has become very important in every- one’s life, even though the importance is often somewhat hidden. For example, if you use Google, a lot of statistics among a lot of other things is used behind the scene. The same applies for face recognition at the airport for passport control. The examples are almost endless. ∗https://en.wikipedia.org/wiki/Statistics 5 Chapter 1. Introduction 1.2. What is “Applied” Statistics? This module is about applied statistics. But what is this thing “applied” statistics? Applied statistics does what the name


SOURCE: Complete lecture notes.pdf
CHUNK ID: 59
TEXT: o not know. But there are also unknown unknowns – the ones we don’t know we don’t know. D

In [ ]:
import re

def format_latex_for_gradio(text):
    # Convert \[ ... \] → $$ ... $$
    text = re.sub(r"\\\[(.*?)\\\]", r"$$\1$$", text, flags=re.DOTALL)

    # Convert \( ... \) → $ ... $
    text = re.sub(r"\\\((.*?)\\\)", r"$\1$", text, flags=re.DOTALL)

    # Fix escaped newlines
    text = text.replace("\\n", "\n")

    return text

In [ ]:
def answer_question(question, k=3):
    retrieved = retrieve_chunks(question, k=k)

    context = "\n\n".join(
        [f"Source: {doc['source']}\n{doc['text']}" for doc in retrieved]
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a domain-specific chatbot for education. "
                    "Answer only using the provided context. "
                    "IMPORTANT:\n"
                    "- Always format mathematical formulas using LaTeX with $$ ... $$\n"
                    "- Never use [ ... ] or plain text for equations\n"
                    "- Use clear, structured explanations\n"
                    "If the answer is not in the context, say: "
                    "'I do not know based on the provided documents.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{question}"
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    answer = format_latex_for_gradio(answer)

    return answer, retrieved

In [ ]:
from IPython.display import display, Markdown

question = "What is “Applied” Statistics?"
answer, retrieved_docs = answer_question(question, k=3)

print("QUESTION:")
print(question)
print("\nANSWER:")
display(Markdown(answer))

QUESTION:
What is “Applied” Statistics?

ANSWER:


Applied statistics is the branch of statistics that focuses on applying statistical methods to real-world problems. It emphasizes critical thinking and practical application rather than requiring extensive prior statistical knowledge. The goal is to use statistical techniques to analyze data and draw conclusions that can inform decision-making in various fields. This module introduces key concepts in applied statistics, including models and sampling, and highlights the difference between classical and Bayesian statistics.

In [ ]:
for doc in retrieved_docs:
    print("=" * 80)
    print("SOURCE:", doc["source"])
    print("CHUNK ID:", doc["chunk_id"])
    print("TEXT:", doc["text"])
    print("\n")

SOURCE: Complete lecture notes.pdf
CHUNK ID: 61
TEXT: t is “Applied” Statistics? This module is about applied statistics. But what is this thing “applied” statistics? Applied statistics does what the name implies: We apply statistics to real everyday problems. The following example should illustrate, what we mean by this. It is more about critical thinking than statistics but has the advantage that practically no prior statistical knowl- edge is required, just a little bit of common sense. Example 1.2.1 The following data are all from 8 and 9 July 2020. It began, like so much else at that time, with a tweet from Donald Trump (see Fig- ure 1.1). Figure 1.1.: T weet from Donald Trump for reopening schools For political reasons, he was determined to reopen schools that had been closed due to the Covid-19 pandemic. Unfortunately , for him, the de


SOURCE: Complete lecture notes.pdf
CHUNK ID: 60
TEXT: tics is the art of data handling . Collecting data used to be a hassle but that is histor

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

def chat_fn(message, history):
    answer, retrieved = answer_question(message, k=3)
    return answer

demo = gr.ChatInterface(
    fn=chat_fn,
    title="Education RAG Chatbot",
    chatbot=gr.Chatbot(
        render_markdown=True,
        latex_delimiters=[
            {"left": "$$", "right": "$$", "display": True},
            {"left": "$", "right": "$", "display": False},
        ],
    ),
)

demo.launch()

/tmp/ipykernel_30453/810281794.py:10: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(
/tmp/ipykernel_30453/810281794.py:10: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 19:25:47 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T19:25:47Z is after 2026-04-01T07:01:35Z


<IPython.core.display.Javascript object>

In [ ]:
evaluation_questions = [
    "What is statistics?",
    "What is the difference between arithmetic mean and expected value?",
    "Why must we distinguish carefully between empirical and theoretical key figures?",
    "Why can relative case numbers be more meaningful than absolute case numbers, and why must we still be careful with them?",
    "What is the formula for the arithmetic mean?",
    "Explain deep learning"  # Not in the PDF
]

results = []

for q in evaluation_questions:
    answer, docs = answer_question(q, k=2)

    context = "\n\n".join(doc["text"] for doc in docs)

    results.append({
        "question": q,
        "answer": answer,
        "context": context
    })

print(results[0]["question"])
print(results[0]["answer"][:500])
print(results[0]["context"][:500])

What is statistics?
Statistics is the discipline that concerns the collection, organization, analysis, interpretation, and presentation of data. Very roughly speaking, statistics is the art of data handling.
tics is the art of data handling . Collecting data used to be a hassle but that is history. With the advent of computer and cheap sophisticated devices to collect data, such as taking pictures with your smart phone, data are everywhere. Therefore, statistics has become very important in every- one’s life, even though the importance is often somewhat hidden. For example, if you use Google, a lot of statistics among a lot of other things is used behind the scene. The same applies for face recognit


In [ ]:
def evaluate_context(question, context):
    prompt = f"""
Estimate how relevant the context is for answering the question.

Question:
{question}

Context:
{context}

Return a number between 0 and 100.

90-100 = highly relevant
70-89 = mostly relevant
40-69 = partially relevant
0-39 = irrelevant
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    import re

    text = response.choices[0].message.content
    number = re.search(r"\d+", text)

    return int(number.group()) if number else 0


def evaluate_faithfulness(question, answer, context):

    if "do not know" in answer.lower() or "not provided" in answer.lower():
        return 100

    prompt = f"""
You are evaluating a RAG system.

Question:
{question}

Answer:
{answer}

Context:
{context}

Task:
Estimate what percentage (0–100) of the answer is supported by the context.

Guidelines:
- 100 = everything in the answer is supported by the context
- 70–90 = mostly supported but includes some added explanation
- 40–60 = partially supported
- 0–30 = mostly unsupported

Return ONLY a number between 0 and 100.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return int(response.choices[0].message.content.strip())

def evaluate_answer_relevance(question, answer):
    prompt = f"""
You are evaluating a RAG system.

Question:
{question}

Answer:
{answer}

Task:
Estimate how well the answer addresses the question.

Guidelines:
- 90-100 = answers clearly and completely
- 70-89 = mostly answers the question
- 40-69 = partially answers
- 0-39 = poor answer or refusal

Return ONLY a number between 0 and 100.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return int(response.choices[0].message.content.strip())

In [ ]:
sample = results[0]

print("QUESTION:")
print(sample["question"])
print("\nANSWER:")
print(sample["answer"])
print("\nCONTEXT PREVIEW:")
print(sample["context"][:1000])

context_score = evaluate_context(sample["question"], sample["context"])
faithfulness_score = evaluate_faithfulness(sample["question"], sample["answer"], sample["context"])
answer_score = evaluate_answer_relevance(sample["question"], sample["answer"])

print("\nSCORES:")
print("Context relevance:", context_score)
print("Faithfulness:", faithfulness_score)
print("Answer relevance:", answer_score)

QUESTION:
What is statistics?

ANSWER:
Statistics is the discipline that concerns the collection, organization, analysis, interpretation, and presentation of data. Very roughly speaking, statistics is the art of data handling.

CONTEXT PREVIEW:
tics is the art of data handling . Collecting data used to be a hassle but that is history. With the advent of computer and cheap sophisticated devices to collect data, such as taking pictures with your smart phone, data are everywhere. Therefore, statistics has become very important in every- one’s life, even though the importance is often somewhat hidden. For example, if you use Google, a lot of statistics among a lot of other things is used behind the scene. The same applies for face recognition at the airport for passport control. The examples are almost endless. ∗https://en.wikipedia.org/wiki/Statistics 5 Chapter 1. Introduction 1.2. What is “Applied” Statistics? This module is about applied statistics. But what is this thing “applied” stat

In [ ]:
evaluations = []

for r in results:
    q = r["question"]
    a = r["answer"]
    c = r["context"]

    evaluations.append({
        "question": q,
        "context_relevance": evaluate_context(q, c),
        "faithfulness": evaluate_faithfulness(q, a, c),
        "answer_relevance": evaluate_answer_relevance(q, a)
    })

In [ ]:
import pandas as pd

df_eval = pd.DataFrame(evaluations)
df_eval

,question,context_relevance,faithfulness,answer_relevance
0,What is statistics?,95,100,90
1,What is the difference between arithmetic mean...,90,90,90
2,Why must we distinguish carefully between empi...,70,90,90
3,Why can relative case numbers be more meaningf...,19,80,90
4,What is the formula for the arithmetic mean?,90,100,100
5,Explain deep learning,20,100,0
